In [1]:
import os
from pathlib import Path
from Bio import SeqIO
import subprocess

# === RUTAS ===
# Carpeta donde están tus archivos fasta individuales
input_folder = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento")  # <-- Modifica esta ruta
# Carpeta de salida para el alineamiento
output_folder = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2")
output_folder.mkdir(parents=True, exist_ok=True)

# Archivo temporal con todas las secuencias unidas
combined_fasta = output_folder / "todas_las_secuencias.fasta"
output_aln = output_folder / "alineamiento.aln"  # Archivo de alineamiento Clustal
output_fasta = output_folder / "alineamiento.fasta"  # Alineamiento en formato FASTA

# === 1. Unir todas las secuencias en un solo FASTA ===
with open(combined_fasta, "w") as outfile:
    for fasta_file in input_folder.glob("*.fasta"):
        for record in SeqIO.parse(fasta_file, "fasta"):
            SeqIO.write(record, outfile, "fasta")

# === 2. Ejecutar Clustal Omega ===
clustalomega_cmd = [
    "clustalo",  # Asegúrate de que Clustal Omega esté en tu PATH
    "-i", str(combined_fasta),
    "-o", str(output_aln),
    "--outfmt", "clu",  # formato Clustal
    "--force"
]

# También guardamos en formato FASTA alineado
clustalomega_fasta_cmd = clustalomega_cmd.copy()
clustalomega_fasta_cmd[clustalomega_fasta_cmd.index("-o") + 1] = str(output_fasta)
clustalomega_fasta_cmd[clustalomega_fasta_cmd.index("--outfmt") + 1] = "fasta"

# Ejecutar
subprocess.run(clustalomega_cmd, check=True)
subprocess.run(clustalomega_fasta_cmd, check=True)

print(f"Alineamiento completado. Archivos generados:\n- {output_aln}\n- {output_fasta}")


Alineamiento completado. Archivos generados:
- C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\alineamiento.aln
- C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\alineamiento.fasta


In [3]:
from Bio import AlignIO, SeqIO
from Bio.Align import MultipleSeqAlignment
from pathlib import Path
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# === RUTA DEL ARCHIVO DE ALINEAMIENTO MULTIPLE ===
alineamiento_fasta = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\alineamiento.fasta")

# === CARGAR EL ALINEAMIENTO ===
alignment = AlignIO.read(alineamiento_fasta, "fasta")

# === DEFINIR LA REGIÓN CONSERVADA (índices Python → desde 417 hasta 813 inclusive) ===
start = 417
end = 814

# === DEFINIR CARPETA DE SALIDA Y ARCHIVO OUTPUT ===
output_folder = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento")
output_folder.mkdir(parents=True, exist_ok=True)
output_fasta = output_folder / "region_conservada_BLAST.fasta"

# === EXTRAER LA REGIÓN, LIMPIAR Y GUARDAR EN FASTA ===
with open(output_fasta, "w") as out_f:
    for record in alignment:
        # Extraer región alineada
        region_seq = record.seq[start:end]
        # Eliminar gaps y caracteres no válidos
        cleaned_seq = "".join([nt for nt in str(region_seq) if nt.upper() in "ATGCNatgcn"])
        # Crear nuevo record limpio
        clean_record = SeqRecord(Seq(cleaned_seq), id=record.id, description=f"region {start+1}-{end}")
        # Guardar
        SeqIO.write(clean_record, out_f, "fasta")

print(f"Zona conservada limpia exportada para BLAST en:\n{output_fasta}")



Zona conservada limpia exportada para BLAST en:
C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\region_conservada_BLAST.fasta


In [4]:
from Bio import SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align.Applications import ClustalOmegaCommandline
from pathlib import Path
import subprocess

# === RUTAS ===
# Tus archivos FASTA individuales
np_influenza_path = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\NP_InfA\data\gene.fna")
np_ebola_path = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\NP_Ebola\ncbi_dataset\data\gene.fna")

# Carpeta de trabajo
output_dir = Path(r"C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\Comparacion_NP")
output_dir.mkdir(parents=True, exist_ok=True)

# === Leer y traducir ambas secuencias (asumimos que son de ADN) ===
records = []
for fasta_path in [np_influenza_path, np_ebola_path]:
    for record in SeqIO.parse(fasta_path, "fasta"):
        prot_seq = record.seq.translate(to_stop=True)
        prot_record = SeqRecord(prot_seq, id=record.id, description="translated")
        records.append(prot_record)

# Guardar ambas proteínas en un único archivo
combined_fasta = output_dir / "NP_combined_proteins.fasta"
SeqIO.write(records, combined_fasta, "fasta")

# === Alinear con Clustal Omega ===
aligned_output = output_dir / "NP_alineado.aln"
clustal_cmd = [
    "clustalo",
    "-i", str(combined_fasta),
    "-o", str(aligned_output),
    "--outfmt", "clu",
    "--force"
]
subprocess.run(clustal_cmd, check=True)

print(f"Alineamiento completado. Resultado en:\n{aligned_output}")


c:\python_env\proyecto_troncal2\Lib\site-packages\Bio\Application\__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(
c:\python_env\proyecto_troncal2\Lib\site-packages\Bio\Seq.py:2879: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


Alineamiento completado. Resultado en:
C:\Users\fgarc\OneDrive\Escritorio\Doctorado\Ramos\1° Semestre\Troncal\proyecto_troncal2\Prueba_alineamiento\Comparacion_NP\NP_alineado.aln
